# Skin Lesion Segmentation with Improved U-Net Models

This notebook trains U-Net, Attention U-Net, U-Net++, and DeepLabV3+ models on Kaggle GPU for binary skin lesion segmentation. After training, download `best_model.pth` and run the local Gradio demo or prediction script.

## 1. Environment Check

Enable GPU in Kaggle Notebook Settings before long training.

In [ ]:
import sys, torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. In Kaggle Notebook Settings, set Accelerator to GPU before formal training.')

## 2. Install Dependencies

If this notebook runs inside a cloned repository, prefer `requirements.txt`. Otherwise install the required packages directly.

In [ ]:
from pathlib import Path
if Path('requirements.txt').exists():
    !pip install -q -r requirements.txt
else:
    !pip install -q albumentations gradio opencv-python-headless pyyaml segmentation-models-pytorch scikit-learn matplotlib tqdm

## 3. Project Code Path

If the project is not already uploaded to the Kaggle working directory, clone it first:

```bash
git clone https://github.com/your-name/medical-image-segmentation.git
cd medical-image-segmentation
```

When using a Kaggle Dataset containing the code, copy or set the working directory to the project root before running the next cells.

## 4. Dataset Paths

Kaggle datasets are mounted under `/kaggle/input/`. Modify the following paths according to your actual dataset layout.

In [ ]:
TRAIN_IMAGES_DIR = '/kaggle/input/your-dataset/train/images'
TRAIN_MASKS_DIR = '/kaggle/input/your-dataset/train/masks'
VAL_IMAGES_DIR = '/kaggle/input/your-dataset/val/images'
VAL_MASKS_DIR = '/kaggle/input/your-dataset/val/masks'

print(TRAIN_IMAGES_DIR)
print(TRAIN_MASKS_DIR)
print(VAL_IMAGES_DIR)
print(VAL_MASKS_DIR)

## 5. Generate Kaggle Runtime Configs

The generated configs keep Kaggle paths in runtime YAML files instead of source code.

In [ ]:
from pathlib import Path
import yaml

def base_config(model_name='unet', image_size=256, batch_size=4, epochs=2):
    return {
        'experiment_name': f'kaggle_runtime_{model_name}',
        'seed': 42,
        'device': 'auto',
        'mixed_precision': True,
        'data': {
            'train_images_dir': TRAIN_IMAGES_DIR,
            'train_masks_dir': TRAIN_MASKS_DIR,
            'val_images_dir': VAL_IMAGES_DIR,
            'val_masks_dir': VAL_MASKS_DIR,
            'image_size': image_size,
        },
        'model': {'model_name': model_name, 'in_channels': 3, 'out_channels': 1},
        'training': {
            'batch_size': batch_size,
            'epochs': epochs,
            'lr': 1e-4,
            'optimizer': 'adam',
            'scheduler': 'none',
            'loss_name': 'bce_dice',
            'num_workers': 2,
            'max_train_batches': 5,
            'max_val_batches': 3,
            'early_stopping': {'enabled': False, 'patience': 3, 'monitor': 'val_dice', 'mode': 'max'},
        },
        'augmentation': {'enabled': True, 'horizontal_flip': True, 'vertical_flip': True, 'rotate': True, 'color_jitter': False},
        'paths': {'output_dir': '/kaggle/working/outputs', 'checkpoint_dir': '/kaggle/working/checkpoints'},
    }

debug_cfg = base_config('unet', image_size=256, batch_size=4, epochs=2)
high_cfg = base_config('unet_plus_plus', image_size=384, batch_size=8, epochs=50)
high_cfg['experiment_name'] = 'kaggle_high_accuracy_unetplusplus_effb3'
high_cfg['model'].update({'encoder_name': 'efficientnet-b3', 'encoder_weights': 'imagenet'})
high_cfg['training'].update({
    'optimizer': 'adamw',
    'scheduler': 'cosine',
    'max_train_batches': None,
    'max_val_batches': None,
    'early_stopping': {'enabled': True, 'patience': 10, 'monitor': 'val_dice', 'mode': 'max'},
})
Path('/kaggle/working').mkdir(parents=True, exist_ok=True)
with open('/kaggle/working/kaggle_debug_runtime.yaml', 'w') as f:
    yaml.safe_dump(debug_cfg, f, sort_keys=False)
with open('/kaggle/working/kaggle_high_accuracy_runtime.yaml', 'w') as f:
    yaml.safe_dump(high_cfg, f, sort_keys=False)
print('/kaggle/working/kaggle_debug_runtime.yaml')
print('/kaggle/working/kaggle_high_accuracy_runtime.yaml')

## 6. Pre-Training Checks

Run dataset check, mask visualization, small-batch overfit, and quick train before long training. If any step fails, do not start formal training.

In [ ]:
!python scripts/check_dataset.py --config /kaggle/working/kaggle_debug_runtime.yaml

In [ ]:
from IPython.display import display
from PIL import Image
import glob
for path in sorted(glob.glob('/kaggle/working/outputs/sanity_check/dataset_overlay_*.png'))[:8]:
    print(path)
    display(Image.open(path))

In [ ]:
!python scripts/overfit_small_batch.py --config /kaggle/working/kaggle_debug_runtime.yaml --epochs 50 --sample-count 8

In [ ]:
from IPython.display import display
from PIL import Image
for path in ['/kaggle/working/outputs/sanity_check/overfit_curves.png']:
    display(Image.open(path))
for path in sorted(glob.glob('/kaggle/working/outputs/sanity_check/overfit_predictions/*overlay.png'))[:4]:
    print(path)
    display(Image.open(path))

In [ ]:
!python scripts/quick_train.py --config /kaggle/working/kaggle_debug_runtime.yaml --epochs 2

## 7. Formal Baseline Training

After all checks pass, train the U-Net baseline. You can also generate a separate runtime YAML without `max_train_batches` for full baseline training.

In [ ]:
# Baseline debug-scale training. For long baseline training, increase epochs and remove max_train_batches/max_val_batches in the YAML.
!python train.py --config /kaggle/working/kaggle_debug_runtime.yaml

## 8. Formal High-Accuracy Training

Recommended configuration: UnetPlusPlus + efficientnet-b3 + ImageNet pretrained weights, image_size 384, batch_size 8, epochs 50, AdamW, cosine scheduler, BCE+Dice loss, mixed precision, early stopping.

In [ ]:
!python train.py --config /kaggle/working/kaggle_high_accuracy_runtime.yaml

## 9. Evaluate Model

In [ ]:
!python evaluate.py --config /kaggle/working/kaggle_high_accuracy_runtime.yaml --checkpoint /kaggle/working/checkpoints/best_model.pth

## 10. Visualize Results

In [ ]:
from IPython.display import display
from PIL import Image
import glob
for path in sorted(glob.glob('/kaggle/working/outputs/samples/*overlay.png'))[:8]:
    print(path)
    display(Image.open(path))
for path in sorted(glob.glob('/kaggle/working/outputs/curves/*.png')):
    print(path)
    display(Image.open(path))

## 11. Saved Outputs and Download

Kaggle saves these files under `/kaggle/working/`:

- `checkpoints/best_model.pth`
- `checkpoints/last_model.pth`
- `outputs/metrics.csv`
- `outputs/experiment_results.csv`
- `outputs/curves/`
- `outputs/samples/`
- `outputs/sanity_check/`

Use the Kaggle Output panel to download them. Put `best_model.pth` into local `checkpoints/`, then run:

```bash
python app.py
python predict.py --image path/to/image.jpg --checkpoint checkpoints/best_model.pth --config configs/unet.yaml
```

Do not hard-code Kaggle dataset paths into source code. Modify only notebook variables or YAML config files. If small-batch overfit fails, quick train returns NaN, or mask overlays are misaligned, stop and fix the dataset or configuration before formal training.